In [ ]:
# ------------------------------------------------------------
# Information Visualisation Coursework
# System A: Spatial Exploration of Road Traffic Accidents
#
# Dataset:
# Leeds Road Traffic Accidents (Kaggle)
# https://www.kaggle.com/datasets/thedevastator/leeds-road-traffic-accidents-2009-2018
#
# Description:
# This notebook implements System A, a map-centric interactive visualisation
# for exploring road traffic accidents in Leeds.
#
# Visualisation Tasks (Munzner, Ch.3 — Actions + Targets):
#
#   T1 — Explore spatial features  [Search -> Explore; Target: Feature]
#        Browse the accident map to discover geographic clusters and hotspot
#        areas where accidents concentrate across Leeds.
#        -> Primary map view shows all accident locations coloured by severity.
#
#   T2 — Discover temporal distribution  [Analyse -> Discover; Target: Distribution]
#        Discover how accident frequency is distributed across hours of the day
#        to identify peak and low-risk time periods.
#        -> Hour histogram updates to show the temporal distribution of the
#           spatially selected accident subset.
#
#   T3 — Select a subset and summarise severity  [Search -> Browse; Query -> Summarise;
#        Target: Distribution]
#        Select a subset of accidents by spatial region and/or hour range, then
#        summarise the severity distribution of that selected subset.
#        *** This task involves selecting a subset of the data. ***
#        -> Spatial brush on the map AND/OR hour brush on the histogram
#           both filter the severity chart simultaneously.
#
# Bidirectional Linking:
#   map_brush  (spatial interval) -- map produces, hour histogram + severity consume
#   hour_brush (temporal interval) -- hour histogram produces, map + severity consume
#   Both selections compose: the map responds to the hour brush, and the hour
#   histogram responds to the map brush. Selecting in either view narrows the other.
#
# Design Decisions (distinguishing A from B and C):
#   D1. Primary view:       A = spatial scatter map
#                           B = time-of-day period bar + hour bar
#                           C = condition stacked bar
#
#   D2. Selection mechanism: A = spatial interval brush + temporal interval brush
#                            B = time-period point click + temporal interval brush
#                                + spatial interval brush
#                            C = categorical point click + temporal interval brush
#                                + spatial interval brush
#
#   D3. Colour in primary:  A = severity (Slight/Serious/Fatal) on the map
#                           B = time-of-day period category on the period bar
#                           C = lighting condition (Daylight/Darkness) on condition bar
#
#   D4. Role of the map:    A = PRIMARY selection surface -- map is the main
#                               interaction point; spatial brush drives linked views
#                           B = BIDIRECTIONAL node -- map responds to time/period
#                               selections AND its own brush feeds back to time chart
#                           C = BIDIRECTIONAL hub -- map responds to two independent
#                               inputs (condition + time) AND its brush updates both
#                               the condition bar and the hour chart
#
#   D5. Severity chart:     A = filtered by spatial region AND hour range (map + hour)
#                           B = filtered by period AND time window AND spatial region
#                           C = filtered by road surface condition AND time window
#
#   D6. Linking direction:  A = spatial <-> temporal (bidirectional: map brush <->
#                               hour brush both feed each other; severity responds
#                               to both)
#                           B = hierarchical cascade with spatial feedback:
#                               period -> time -> space, plus space -> time
#                           C = dual bidirectional: condition <-> space and
#                               time <-> space (map brush updates both condition
#                               bar and hour chart)
#
# Libraries used:
# pandas -- data processing
# altair -- interactive visualisation
# ------------------------------------------------------------

In [ ]:
# !pip install altair pandas

In [ ]:
import pandas as pd
import altair as alt

In [ ]:
alt.data_transformers.disable_max_rows()
df = pd.read_csv("data\\data.csv")
df = df[df["has_coords"] == True].copy()
df = df.dropna(subset=["hour", "severity"])
df["hour"] = df["hour"].astype(int)
df["severity"] = df["severity"].astype(str)
print(df.shape)
df.head()

In [ ]:
df.columns

In [ ]:
# ---- Bidirectional selections -------------------------------------------
#
# map_brush  : drawn on the map  -> filters hour histogram + severity chart
# hour_brush : drawn on the hour bar -> filters the map + severity chart
#
# When either brush is empty all data passes through (interval default).

map_brush  = alt.selection_interval(name="map_brush")
hour_brush = alt.selection_interval(name="hour_brush", encodings=["x"])

In [ ]:
# ---- T1 + T3: Spatial Map (primary view) --------------------------------
# Produces:  map_brush  (spatial selection -> drives hour heatmap + severity)
# Consumes:  hour_brush (temporal selection -> shows only accidents in range)
#
# D4: Map is the PRIMARY selection surface in System A.
# Fixed: explicit domain prevents map rescaling when brush is active.

xmin, xmax = int(df["easting"].min()),  int(df["easting"].max())
ymin, ymax = int(df["northing"].min()), int(df["northing"].max())

map_chart = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.40)
    .transform_filter(hour_brush)
    .encode(
        x=alt.X("easting:Q",  scale=alt.Scale(domain=[xmin, xmax]), title="Easting"),
        y=alt.Y("northing:Q", scale=alt.Scale(domain=[ymin, ymax]), title="Northing"),
        color=alt.Color(
            "severity:N",
            scale=alt.Scale(
                domain=["Slight", "Serious", "Fatal"],
                range=["#4CAF50", "#FC8D59", "#D73027"]
            )
        ),
        tooltip=["severity", "accident_id", "num_casualties", "accident_date", "time_24h"]
    )
    .add_params(map_brush)
    .properties(
        width=500, height=400,
        title="T1 / T3 — Spatial Distribution (drag to select region)"
    )
)

In [ ]:
# ---- T2: Weekday × Hour Heatmap (bidirectional) -------------------------
# Consumes:  map_brush  (spatial selection -> reshapes heatmap cells)
# Produces:  hour_brush (hour interval -> drives map + severity)
#
# Color encodes accident density per day-hour cell.
# Opacity dims columns outside the hour brush so the selected range stands out.
# Dragging left-right across hours produces the same hour-interval selection
# as the original bar chart (hour_brush encodings=["x"] unchanged).

WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

hour_chart = (
    alt.Chart(df)
    .transform_filter(map_brush)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["hour", "weekday"]
    )
    .mark_rect()
    .encode(
        x=alt.X("hour:O", title="Hour of Day"),
        y=alt.Y("weekday:N", sort=WEEKDAY_ORDER, title="Day of Week"),
        color=alt.Color(
            "accident:Q",
            scale=alt.Scale(scheme="blues"),
            title="Accidents"
        ),
        opacity=alt.condition(hour_brush, alt.value(1.0), alt.value(0.15)),
        tooltip=[
            alt.Tooltip("hour:O",     title="Hour"),
            alt.Tooltip("weekday:N",  title="Day"),
            alt.Tooltip("accident:Q", title="Accidents")
        ]
    )
    .add_params(hour_brush)
    .properties(
        width=400, height=180,
        title="T2 — Accidents by Weekday & Hour (drag hours to select)"
    )
)

In [ ]:
# ---- T3: Severity Chart (filtered by BOTH brushes) ----------------------
# Consumes: map_brush AND hour_brush
# Shows the severity profile of the doubly-filtered accident subset.
# D5: filtered by spatial region AND hour range (unique to System A).

severity_chart = (
    alt.Chart(df)
    .transform_filter(map_brush)           # filter by spatial selection
    .transform_filter(hour_brush)          # filter by temporal selection
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["severity"]
    )
    .mark_bar()
    .encode(
        x=alt.X("severity:N", sort=["Fatal", "Serious", "Slight"], title="Severity"),
        y=alt.Y("accident:Q", title="Number of Accidents"),
        color=alt.Color("severity:N", legend=None)
    )
    .properties(
        width=250, height=200,
        title="T3 — Severity of Selected Subset"
    )
)

In [ ]:
# ---- Layout -------------------------------------------------------------
# Map (top) drives both linked charts below it via map_brush.
# Hour chart also drives the map and severity via hour_brush (bidirectional).

dashboard = (
    (map_chart & (hour_chart | severity_chart))
    .configure_axis(gridOpacity=0.30)
    .configure_view(stroke=None)
)
dashboard